In [1]:
print("hello")

hello


In [2]:
print("all ok")

all ok


In [3]:
import uuid
from dataclasses import dataclass
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)
from langgraph.runtime import Runtime

In [4]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "openai:gpt-5-nano",
    temperature=0
)

c:\Users\Sudheer.Gundra\AppData\Local\anaconda3\envs\sudheer_agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [8]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

In [10]:
@dataclass
class Context:
    user_id: str


In [11]:
Context.user_id=str(uuid.uuid4())

In [12]:
Context.user_id

'2b2efe70-7b0d-4188-8247-4c57bfcc8a67'

In [ ]:
def assistant_node(
    state: MessagesState,
    runtime: Runtime[Context]
):
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "memories")
    user_message = state["messages"][-1].content
    
    if user_message.lower().startswith("remember:"):
        memory = user_message[len("remember:"):].strip()
        runtime.store.put(
            namespace,
            str(uuid.uuid4()),
            {
                "data": memory
            }
        )
        print(
            f"[MEMORY SAVED]: {memory}"
        )
        
    memories = runtime.store.search(namespace)
    memory_text = "\n".join(
        memory.value["data"]
        for memory in memories
    )
    if not memory_text:
        memory_text = "No saved memories."
        
    
    system_prompt = f"""
    You are a helpful assistant.

    Long-term memories about this user:

    {memory_text}
    """


    response = model.invoke(
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *state["messages"]
        ]
    )
    return {
        "messages": [response]
    }

In [19]:
builder = StateGraph(
    MessagesState,
    context_schema=Context
)


In [20]:
builder.add_node(
    "assistant",
    assistant_node
)

In [21]:
builder.add_edge(
    START,
    "assistant"
)

builder.add_edge(
    "assistant",
    END
)

In [22]:
graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [29]:
def chat(
    message,
    user_id,
    thread_id
):

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        config={
            "configurable": {
                "thread_id": thread_id
            }
        },

        context=Context(
            user_id=user_id
        )
    )


    answer = result["messages"][-1].content

    print("\nUSER:")
    print(message)

    print("\nASSISTANT:")
    print(answer)

    print("\n" + "=" * 60)

In [25]:
chat(
    message="My current project is Agentic RAG.",
    user_id="maha_123",
    thread_id="thread_1"
)


USER:
My current project is Agentic RAG.

ASSISTANT:
Great! Could you share more details about Agentic RAG? For example, what does the project involve, what are its objectives, and how can I assist you with it?



In [30]:
chat(
    message="What is my current project?",
    user_id="maha_123",
    thread_id="thread_1"
)


USER:
What is my current project?

ASSISTANT:
Your current project is Agentic RAG.



In [31]:
chat(
    message="Remember: my favorite programming language is Python.",
    user_id="maha_123",
    thread_id="thread_1"
)

[MEMORY SAVED]: my favorite programming language is Python.

USER:
Remember: my favorite programming language is Python.

ASSISTANT:
I’ve already noted that your favorite programming language is Python. If there’s anything specific you want to do with Python or your Agentic RAG project, feel free to ask!



In [32]:
chat(
    message="What is my favorite programming language?",
    user_id="maha_123",
    thread_id="thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Python.



                 maha_123
                     ↓
              InMemoryStore
                     ↓
      "favorite language = Python"
                     ↓
             ┌───────┴───────┐
             ↓               ↓
         thread_1         thread_2

In [33]:
chat(
    message="Remember: my favorite language is Python.",
    user_id="maha_123",
    thread_id="maha_thread_1"
)

[MEMORY SAVED]: my favorite language is Python.

USER:
Remember: my favorite language is Python.

ASSISTANT:
Got it! I've remembered that your favorite language is Python. If there's anything related to Python you'd like help with, just let me know!



In [35]:
chat(
    message="Remember: my favorite language is Java.",
    user_id="rahul_456",
    thread_id="rahul_thread_1"
)

[MEMORY SAVED]: my favorite language is Java.

USER:
Remember: my favorite language is Java.

ASSISTANT:
Got it! I've remembered that your favorite language is Java. If you need any help with Java or anything else, just let me know!



users
│
├── maha_123
│      └── memories
│           └── favorite language = Python
│
└── rahul_456
       └── memories
            └── favorite language = Java

In [36]:
chat(
    message="What is my favorite programming language?",
    user_id="rahul_456",
    thread_id="rahul_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Java.



In [37]:
chat(
    message="What is my favorite programming language?",
    user_id="maha_123",
    thread_id="maha_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
Your favorite programming language is Python.



In [39]:
user_id = "maha_123"

In [40]:
namespace = (
    "users",
    user_id,
    "memories"
)


In [41]:
memories = store.search(namespace)

for memory in memories:
    print(memory.value)

{'data': 'my favorite programming language is Python.'}
{'data': 'my favorite programming language is Python.'}
{'data': 'my favorite language is Python.'}


In [42]:
memories = store.search(
    ("users", "maha_123", "memories")
)

for memory in memories:
    print(memory.value["data"])

my favorite programming language is Python.
my favorite programming language is Python.
my favorite language is Python.
